# Experiment: LLM-based Cleaning with BgGPT-Gemma-3

**Goal:** Compare three versions of the same articles:
1. `raw_text` — original scraped text (from raw JSON files)
2. `rule_based_text` — cleaned by the existing pipeline (from `articles_clean.parquet`)
3. `bggpt_text` — cleaned by BgGPT-Gemma-3-4B-IT

**Period:** 2026-05-14 to 2026-05-21 (1 week)  
**Sample:** ~300 articles, stratified by source

In [13]:
import os
import json
import glob
import pandas as pd
import re
from tqdm.notebook import tqdm
import ollama

RAW_DIR = "../data/raw"
PARQUET_PATH = "../data/processed/articles_clean.parquet"
START_DATE = "2026-05-14"
END_DATE = "2026-05-21"

## Step 1: Load Raw Articles (May 14–21)

Each day folder contains multiple JSON files — one per fetch batch per source. Each file is a single article dict. We walk all 7 day folders, load every JSON, and collect them into a flat list.

In [2]:
records = []

day_folders = sorted([
    d for d in os.listdir(RAW_DIR)
    if START_DATE <= d <= END_DATE
])

for day in day_folders:
    json_files = glob.glob(os.path.join(RAW_DIR, day, "*.json"))
    for fpath in json_files:
        with open(fpath, "r", encoding="utf-8") as f:
            article = json.load(f)
        records.append(article)

df_raw = pd.DataFrame(records)
df_raw["published_at"] = pd.to_datetime(df_raw["published_at"], utc=True, errors="coerce")

print(f"Total raw articles loaded: {len(df_raw)}")
print(df_raw["source"].value_counts())

Total raw articles loaded: 12070
source
monitor         3150
24chasa         2164
actualno        1349
fakti           1145
bta             1060
standartnews     927
nova             862
vesti            675
segabg           392
banker           240
economic         106
Name: count, dtype: int64


## Step 2: Load Rule-Based Cleaned Articles (same period)

Filter the cleaned parquet to the same week. We'll join on `url` to match raw ↔ cleaned pairs.

In [3]:
df_clean = pd.read_parquet(PARQUET_PATH)
df_clean["published_at_dt"] = pd.to_datetime(df_clean["published_at_dt"], utc=True)

df_clean_week = df_clean[
    (df_clean["published_at_dt"] >= START_DATE) &
    (df_clean["published_at_dt"] <= END_DATE)
].copy()

print(f"Cleaned articles in week: {len(df_clean_week)}")
print(df_clean_week["source"].value_counts())

Cleaned articles in week: 7932
source
24chasa         1817
actualno        1085
fakti            985
bta              911
standartnews     792
monitor          615
nova             600
vesti            581
segabg           252
banker           203
economic          91
Name: count, dtype: int64


## Step 3: Join Raw ↔ Cleaned on URL + Stratified Sample

We join on `url` to get matched pairs where we have both raw text and rule-based cleaned text. Then we sample ~300 articles proportionally per source so all 11 sources are represented.

In [4]:
df_raw_dedup = df_raw.drop_duplicates(subset="url")

df_merged = df_raw_dedup.merge(
    df_clean_week[["url", "full_text", "word_count"]].rename(columns={
        "full_text": "clean_text",
        "word_count": "clean_word_count"
    }),
    on="url",
    how="inner"
).rename(columns={"full_text": "raw_text"})

print(f"Matched pairs: {len(df_merged)}")
print(df_merged["source"].value_counts())

# Stratified sample of 300 — loop avoids pandas 3.0 groupby/apply key-exclusion issue
SAMPLE_SIZE = 300
total = len(df_merged)
dfs = []
for source, group in df_merged.groupby("source"):
    n = max(1, round(SAMPLE_SIZE * len(group) / total))
    dfs.append(group.sample(n=min(len(group), n), random_state=42))

df_sample = pd.concat(dfs).reset_index(drop=True)
print(f"\nSample size: {len(df_sample)}")
print(df_sample["source"].value_counts())

Matched pairs: 7932
source
24chasa         1817
actualno        1085
fakti            985
bta              911
standartnews     792
monitor          615
nova             600
vesti            581
segabg           252
banker           203
economic          91
Name: count, dtype: int64

Sample size: 300
source
24chasa         69
actualno        41
fakti           37
bta             34
standartnews    30
monitor         23
nova            23
vesti           22
segabg          10
banker           8
economic         3
Name: count, dtype: int64


# Step 4 - Load the model

In [7]:
models = [m.model for m in ollama.list().models]
print("Available models:", models)

Available models: ['bggpt:latest']


# Step 5 - Cleaning function

In [10]:
SYSTEM_PROMPT = (
    "Ти си инструмент за почистване на новинарски текстове на български. "
    "Ще получиш текст от сайта {source}. "
    "Премахни всичко, което не е основното съдържание: реклами, навигация, "
    "footer, header, социални бутони, повтарящи се фрази, препоръчани статии. "
    "Върни САМО валиден JSON без никакъв друг текст:\n"
    '{{"cleaned_text": "<основен текст>", "confidence": <1-5>}}'
)


def clean_with_bggpt(raw_text: str, source: str) -> dict:
    response = ollama.chat(
        model="bggpt",
        messages=[
            {
                "role": "user",
                "content": SYSTEM_PROMPT.format(source=source)
                + f"\n\nТекст:\n{raw_text[:3000]}",
            }
        ],
    )
    decoded = response.message.content
    match = re.search(r"\{.*\}", decoded, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return {"cleaned_text": decoded.strip(), "confidence": None}

In [11]:
test = df_sample.iloc[0]
result = clean_with_bggpt(test["raw_text"], test["source"])
print("Source:", test["source"])
print("Confidence:", result.get("confidence"))
print("\nRAW:\n", test["raw_text"][:500])
print("\nBgGPT CLEANED:\n", result.get("cleaned_text", "")[:500])
print("\nRULE-BASED CLEANED:\n", test["clean_text"][:500])

Source: 24chasa
Confidence: 5

RAW:
 Германският министър на отбраната Борис Писториус днес заяви, че Берлин все още не е получил никакви обвързващи планове от администрацията на Тръмп относно изтеглянето на американски войски от Германия, предаде ДПА, цитирана от БТА.
"Все още няма наистина надеждно потвърждение за това", каза министърът след заседание на парламентарната комисия по отбрана в Берлин.
Това, което Германия е чула, каза той, е, че броят на американските бригади, разположени в Европа, може да бъде намален с една формац

BgGPT CLEANED:
 Германският министър на отбраната Борис Писториус днес заяви, че Берлин все още не е получил никакви обвързващи планове от администрацията на Тръмп относно изтеглянето на американски войски от Германия, предаде ДПА, цитирана от БТА. "Все още няма наистина надеждно потвърждение за това", каза министърът след заседание на парламентарната комисия по отбрана в Берлин. Това, което Германия е чула, каза той, е, че броят на американските бригади, р

In [15]:
os.makedirs("../data/experiments", exist_ok=True)

results = []
for _, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
    result = clean_with_bggpt(row["raw_text"], row["source"])
    results.append(
        {
            "url": row["url"],
            "source": row["source"],
            "raw_text": row["raw_text"],
            "rule_based_text": row["clean_text"],
            "bggpt_text": result.get("cleaned_text", ""),
            "bggpt_confidence": result.get("confidence"),
        }
    )

df_results = pd.DataFrame(results)
df_results.to_parquet(
    "../data/experiments/bggpt_cleaning_experiment.parquet", index=False
)
print(f"Saved {len(df_results)} results.")

  0%|          | 0/300 [00:00<?, ?it/s]

Saved 300 results.


In [20]:
import textwrap


def jaccard(text_a: str, text_b: str) -> float:
    a, b = set(text_a.lower().split()), set(text_b.lower().split())
    if not a and not b:
        return 1.0
    return len(a & b) / len(a | b)


def similarity_flag(score: float) -> str:
    if score >= 0.85:
        return "✦ VERY SIMILAR"
    if score >= 0.60:
        return "~ SOMEWHAT DIFFERENT"
    return "✗ VERY DIFFERENT"


def display_comparison(row, width=80, words_per_line=20):
    sim = jaccard(row["rule_based_text"], row["bggpt_text"])
    flag = similarity_flag(sim)

    def wrap(text):
        words = text.split()
        lines = [
            " ".join(words[i : i + words_per_line])
            for i in range(0, len(words), words_per_line)
        ]
        return "\n".join(lines)  # show first 10 lines

    print("=" * width)
    print(
        f"SOURCE: {row['source']}  |  CONFIDENCE: {row['bggpt_confidence']}  |  SIMILARITY: {sim:.2f}  {flag}"
    )
    print("-" * width)
    print("RULE-BASED:")
    print(wrap(str(row["rule_based_text"])))
    print("\nBgGPT:")
    print(wrap(str(row["bggpt_text"])))
    print()

In [22]:
for _, row in df_results.head(5).iterrows():
    display_comparison(row)

SOURCE: 24chasa  |  CONFIDENCE: 5.0  |  SIMILARITY: 0.98  ✦ VERY SIMILAR
--------------------------------------------------------------------------------
RULE-BASED:
Германският министър на отбраната Борис Писториус днес заяви, че Берлин все още не е получил никакви обвързващи планове от администрацията
на Тръмп относно изтеглянето на американски войски от Германия, предаде ДПА, цитирана от БТА. "Все още няма наистина надеждно потвърждение
за това", каза министърът след заседание на парламентарната комисия по отбрана в Берлин. Това, което Германия е чула, каза той,
е, че броят на американските бригади, разположени в Европа, може да бъде намален с една формация. "До каква степен това
ще засегне войските, разположени в Германия, тепърва предстои да се види", добави Писториус. Ден по-рано американският генерал и върховен главнокомандващ
на НАТО Алексъс Гринкевич потвърди, че САЩ засега няма да разполагат оръжия с голям обсег в Германия. Планираното по-рано разполагане
на т. нар. батальон 

In [23]:
import matplotlib.pyplot as plt
import numpy as np

df_results = pd.read_parquet("../data/experiments/bggpt_cleaning_experiment.parquet")

df_results["raw_words"] = df_results["raw_text"].str.split().str.len()
df_results["rule_words"] = df_results["rule_based_text"].str.split().str.len()
df_results["bggpt_words"] = df_results["bggpt_text"].str.split().str.len()
df_results["jaccard"] = df_results.apply(
    lambda r: jaccard(str(r["rule_based_text"]), str(r["bggpt_text"])), axis=1
)

In [24]:
summary = (
    pd.DataFrame(
        {
            "": ["Raw", "Rule-based", "BgGPT"],
            "Avg words": [
                df_results["raw_words"].mean(),
                df_results["rule_words"].mean(),
                df_results["bggpt_words"].mean(),
            ],
            "Median words": [
                df_results["raw_words"].median(),
                df_results["rule_words"].median(),
                df_results["bggpt_words"].median(),
            ],
            "Reduction vs raw (%)": [
                0,
                (
                    (df_results["raw_words"] - df_results["rule_words"])
                    / df_results["raw_words"]
                    * 100
                ).mean(),
                (
                    (df_results["raw_words"] - df_results["bggpt_words"])
                    / df_results["raw_words"]
                    * 100
                ).mean(),
            ],
        }
    )
    .set_index("")
    .round(1)
)

In [25]:
print(summary)

            Avg words  Median words  Reduction vs raw (%)
                                                         
Raw             383.1         276.0                   0.0
Rule-based      341.6         252.5                   8.6
BgGPT           248.0         216.5                  19.7


### Word Count Analysis

| | Avg words | Median words | Reduction vs raw |
|---|---|---|---|
| Raw | 383.1 | 276.0 | — |
| Rule-based | 341.6 | 252.5 | 8.6% |
| BgGPT | 248.0 | 216.5 | 19.7% |

**Key observations:**

- BgGPT removes **2.3× more text** than the rule-based cleaner (19.7% vs 8.6% reduction).
- The raw dataset is right-skewed (mean 383 vs median 276), indicating a subset of very long articles.
- BgGPT normalizes article lengths more aggressively — its mean/median gap is smaller,
  suggesting it applies stricter trimming on longer articles.
- The rule-based pipeline targets structural noise (navigation, boilerplate),
  while BgGPT appears to apply a broader definition of "main content",
  potentially removing author bylines, related article links, and source attributions.

**Open question:** the larger reduction by BgGPT does not by itself indicate better cleaning —
it may reflect over-cleaning. Agreement analysis (Jaccard similarity) is needed to assess
where the two approaches diverge.


In [28]:
print("=== Rule-based vs BgGPT Agreement ===")
print(f"Mean Jaccard similarity:   {df_results['jaccard'].mean():.3f}")
print(f"Median Jaccard similarity: {df_results['jaccard'].median():.3f}")
print("\nSimilarity breakdown:")
print(f"  Very similar  (≥0.85): {(df_results['jaccard'] >= 0.85).sum()} articles")
print(
    f"  Somewhat diff (0.60–0.85): {((df_results['jaccard'] >= 0.60) & (df_results['jaccard'] < 0.85)).sum()} articles"
)
print(f"  Very different (<0.60): {(df_results['jaccard'] < 0.60).sum()} articles")

print("\n=== BgGPT Confidence ===")
print(df_results["bggpt_confidence"].value_counts().sort_index())
print(
    f"Mean confidence: {pd.to_numeric(df_results['bggpt_confidence'], errors='coerce').mean():.2f} / 5"
)

=== Rule-based vs BgGPT Agreement ===
Mean Jaccard similarity:   0.835
Median Jaccard similarity: 0.931

Similarity breakdown:
  Very similar  (≥0.85): 198 articles
  Somewhat diff (0.60–0.85): 59 articles
  Very different (<0.60): 43 articles

=== BgGPT Confidence ===
bggpt_confidence
1.0      1
3.0     12
4.0     89
5.0    153
Name: count, dtype: int64
Mean confidence: 4.54 / 5


### Jaccard Similarity & Confidence Analysis

| Metric | Value |
|---|---|
| Mean Jaccard similarity | 0.835 |
| Median Jaccard similarity | 0.931 |
| Very similar (≥ 0.85) | 198 articles (66%) |
| Somewhat different (0.60–0.85) | 59 articles (20%) |
| Very different (< 0.60) | 43 articles (14%) |
| Mean BgGPT confidence | 4.54 / 5 |

**Key observations:**

- The high median (0.931) indicates that both methods broadly agree on what constitutes
  main content for the majority of articles.
- The mean (0.835) is pulled down by a tail of 43 highly divergent cases,
  suggesting the disagreement is concentrated rather than uniform.
- BgGPT reports high confidence (4.54/5) across all sources,
  including the cases where it diverges most from the rule-based output.
- 15% of articles failed to return valid JSON — the model produced free-text
  instead of the requested structured output.

**Qualitative finding — structured documents:**
Manual inspection of the most divergent cases (Jaccard < 0.10) reveals that BgGPT
struggles with non-narrative content such as official bulletins and structured reports
(e.g. road condition updates). For these article types, BgGPT incorrectly applies
summarization logic, discarding structured data that constitutes the actual content.
The rule-based pipeline, being structure-agnostic, preserves this content correctly.


In [27]:
per_source = (
    df_results.groupby("source")
    .agg(
        articles=("url", "count"),
        mean_jaccard=("jaccard", "mean"),
        mean_confidence=(
            "bggpt_confidence",
            lambda x: pd.to_numeric(x, errors="coerce").mean(),
        ),
        avg_raw_words=("raw_words", "mean"),
        avg_rule_words=("rule_words", "mean"),
        avg_bggpt_words=("bggpt_words", "mean"),
    )
    .round(3)
    .sort_values("mean_jaccard")
)

print(per_source)

              articles  mean_jaccard  mean_confidence  avg_raw_words  \
source                                                                 
economic             3         0.615            4.000        427.000   
segabg              10         0.660            4.286        527.000   
vesti               22         0.771            4.421        494.227   
nova                23         0.787            4.571        284.174   
actualno            41         0.809            4.289        340.561   
bta                 34         0.817            4.724        307.059   
24chasa             69         0.834            4.771        449.855   
fakti               37         0.897            4.323        508.243   
standartnews        30         0.897            4.483        322.400   
monitor             23         0.900            4.739        232.304   
banker               8         0.950            4.857        211.625   

              avg_rule_words  avg_bggpt_words  
source         

In [32]:
df_divergent = df_results[df_results["jaccard"] < 0.60].sort_values("jaccard")
for _, row in df_divergent.head(3).iterrows():
    display_comparison(row)

SOURCE: 24chasa  |  CONFIDENCE: nan  |  SIMILARITY: 0.07  ✗ VERY DIFFERENT
--------------------------------------------------------------------------------
RULE-BASED:
Информация за състоянието на републиканските пътища към 17:30 часа на 17.05.2026 г. I. МЕТЕОРОЛОГИЧНА ОБСТАНОВКА: През нощта облачността ще се
разкъсва и намалява до предимно ясно време. Ще духа до умерен вятър от запад-северозапад. Утре 18.05.2026 г.облачността ще е променлива,
след обяд - често значителна, но само на отделни места в планинските райони ще превали слабо. Ще духа умерен до
силен вятър от запад-северозапад. Максималните температури ще са между 19° и 24°, в София - около 19°. II. СЪСТОЯНИЕ НА
АВТОМАГИСТРАЛИТЕ: АМ „Тракия": Област София: Движението при Мухово от км 44+400 до км 46+480 в посока Бургас се осъществява с
повишено внимание поради обезопасяване на пътния участък в аварийната лента. Срок до 31.12.2026 г.; Област Пазарджик: Движението в участъка от
км 66+750 до км 80+050 посока Бургас се осъществява

In [31]:
row = df_results[df_results["jaccard"] < 0.60].sort_values("jaccard").iloc[0]
print("BgGPT output:\n", row["bggpt_text"][:1000])
print("\nConfidence:", row["bggpt_confidence"])

BgGPT output:
 ```json
{
  "cleaned_text": "Информация за състоянието на републиканските пътища към 17:30 часа на 17.05.2026 г. I. МЕТЕОРОЛОГИЧНА ОБСТАНОВКА: През нощта облачността ще се разкъсва и намалява до предимно ясно време. Ще духа до умерен вятър от запад-северозапад. Утре 18.05.2026 г.облачността ще е променлива, след обяд - често значителна, но само на отделни места в планинските райони ще превали слабо. Ще духа умерен до силен вятър от запад-северозапад. Максималните температури ще са между 19° и 24°, в София - около 19°. II. СЪСТОЯНИЕ НА АВТОМАГИСТРАЛИТЕ: АМ „Тракия": Област София: Движението при Мухово от км 44+400 до км 46+480 в посока Бургас се осъществява с повишено внимание поради обезопасяване на пътния участък в аварийната лента. Срок до 31.12.2026 г.; Област Пазарджик: Движението в участъка от км 66+750 до км 80+050 посока Бургас се осъществява с повишено внимание и с ограничение на скоростта до 100 км/ч поради неравности. Участъкът е сигнализиран с пътни знаци. Сро